# Gold Layer - Executive Summary Fact Table

## Purpose
Provide a single-row executive dashboard with high-level KPIs for business leadership: total orders, customers, products, revenue, basket size, and reorder rate.

## Type
**Fact Table** (materialized, aggregated metrics)

## Input
* **Source:** `big_data.silver.orders` (3.3M rows)
* **Source:** `big_data.silver.products_enriched` (49.7K rows)
* **Source:** `big_data.silver.order_products` (33.8M rows)

## Output
* **Target:** `big_data.gold.ft_executive_summary`
* **Rows:** 1 (aggregated)
* **Primary Key:** None (single aggregated row)

## Transformations

### Step 1: Load Silver Tables
* Load orders, products_enriched, and order_products
* Join order_products with products for pricing

### Step 2: Calculate Executive KPIs
* **Total Orders:** COUNT(DISTINCT order_id)
* **Total Customers:** COUNT(DISTINCT user_id)
* **Total Products:** COUNT(DISTINCT product_id)
* **Total Items Sold:** COUNT(*) from order_products
* **Estimated Total Revenue:** SUM(price_usd) across all items
* **Avg Basket Size:** AVG(items per order)
* **Avg Order Value:** AVG(order revenue)
* **Overall Reorder Rate:** % of items that were reordered
* Add `_gold_timestamp`

## Data Quality Validations

### Technical Validations
* Row count = 1 (single aggregated row)
* NOT NULL on all KPI columns
* All numeric values >= 0

### Business Validations
* Total orders > 3M (expected from Silver)
* Total customers > 200K
* Reorder rate between 50-70%
* Avg basket size between 8-12 items

## Why Materialized?
* ❌ Heavy query (scan 33.8M items)
* ✅ Dashboard principal (accessed frequently)
* ✅ Pre-computed KPIs

## Persistence
Only persists to Delta table if **all validations pass**.

## Execution
Run all cells sequentially. Expected runtime: ~3-5 minutes.

In [0]:
%run ../UTILS/data_quality_checks

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Source tables
source_tables = {
    "orders": "orders",
    "products": "products_enriched",
    "order_products": "order_products"
}

# Target table (fact table with ft_ prefix)
target_table = "ft_executive_summary"

# Expected metrics (for validation)
expected_metrics = {
    "min_total_orders": 3_000_000,
    "min_total_customers": 200_000,
    "min_reorder_rate": 50.0,
    "max_reorder_rate": 70.0,
    "min_avg_basket_size": 8.0,
    "max_avg_basket_size": 12.0
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Target: {gold_schema}.{target_table}")
print(f"  Source tables: {list(source_tables.keys())}")

In [0]:
print("Step 1: Loading Silver tables and joining...")

# Load Silver tables
orders = spark.table(f"{silver_schema}.{source_tables['orders']}")
products = spark.table(f"{silver_schema}.{source_tables['products']}")
order_products = spark.table(f"{silver_schema}.{source_tables['order_products']}")

print(f"  Orders loaded: {orders.count():,} rows")
print(f"  Products loaded: {products.count():,} rows")
print(f"  Order-Products loaded: {order_products.count():,} rows")

# Join order_products with products to get pricing
order_products_enriched = order_products \
    .join(products.select("product_id", "price_usd"), "product_id", "left")

print(f"\n  Enriched order_products with pricing: {order_products_enriched.count():,} rows")

In [0]:
print("Step 2: Calculating executive KPIs...")

# Calculate order-level metrics first
order_level_metrics = orders \
    .join(order_products_enriched, "order_id") \
    .groupBy("order_id") \
    .agg(
        F.count("product_id").alias("basket_size"),
        F.round(F.sum("price_usd"), 2).alias("estimated_order_value_usd"),
        F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    )

# Calculate executive summary
executive_summary_df = order_level_metrics.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.avg("basket_size"), 2).alias("avg_basket_size"),
    F.round(F.avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
    F.round(F.sum("estimated_order_value_usd"), 2).alias("estimated_total_revenue_usd"),
    F.sum("reordered_items").alias("total_reordered_items")
)

# Add global metrics
total_customers = orders.select("user_id").distinct().count()
total_products = products.select("product_id").distinct().count()
total_items = order_products.count()

executive_summary_gold = executive_summary_df \
    .withColumn("total_customers", F.lit(total_customers)) \
    .withColumn("total_products", F.lit(total_products)) \
    .withColumn("total_items_sold", F.lit(total_items)) \
    .withColumn(
        "overall_reorder_rate",
        F.round(
            (F.col("total_reordered_items") / F.col("total_items_sold")) * 100,
            2
        )
    ) \
    .withColumn("_gold_timestamp", F.current_timestamp())

print("  Executive KPIs calculated")
print("\nPreview:")
executive_summary_gold.show(1, truncate=False, vertical=True)

In [0]:
print_validation_header("executive_summary - Technical Validations")

total_rows = executive_summary_gold.count()
print(f"\nTotal rows: {total_rows:,}")
print(f"Expected: 1 row (aggregated summary)\n")

# Initialize validation flag
validation_passed_technical = True

# 1. Row count check (must be exactly 1)
if total_rows != 1:
    status = "FAIL"
    msg = f"Expected 1 row, got {total_rows}"
    validation_passed_technical = False
else:
    status = "PASS"
    msg = "Exactly 1 aggregated row"
print_check_result("ROW COUNT (=1)", status, msg)

# 2. NOT NULL checks (all KPI columns)
kpi_columns = [
    "total_orders", "total_customers", "total_products", 
    "total_items_sold", "estimated_total_revenue_usd",
    "avg_basket_size", "avg_order_value_usd", "overall_reorder_rate"
]
status, failed, msg = check_not_null(executive_summary_gold, kpi_columns)
print_check_result(f"NOT NULL ({len(kpi_columns)} KPIs)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

# 3. Non-negative checks (all metrics should be >= 0)
print("\n3. Non-negative Validation:")
negative_count = 0
for col_name in kpi_columns:
    col_value = executive_summary_gold.select(col_name).first()[0]
    if col_value is not None and col_value < 0:
        print(f"  ✗ {col_name}: {col_value} (NEGATIVE)")
        negative_count += 1
        validation_passed_technical = False

if negative_count == 0:
    status = "PASS"
    msg = "All metrics are non-negative"
else:
    status = "FAIL"
    msg = f"{negative_count} metrics are negative"
print_check_result("NON-NEGATIVE (all KPIs >= 0)", status, msg, negative_count)

print("\n" + "="*60)
if validation_passed_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("executive_summary - Business Validations")

# Initialize business validation flag
validation_passed_business = True

# Get the summary row
summary_row = executive_summary_gold.first()

# 1. Total orders validation
print("\n1. Business Rule - Total Orders:")
total_orders = summary_row["total_orders"]
if total_orders >= expected_metrics["min_total_orders"]:
    status = "PASS"
    msg = f"Total orders ({total_orders:,}) >= {expected_metrics['min_total_orders']:,}"
else:
    status = "FAIL"
    msg = f"Total orders ({total_orders:,}) < {expected_metrics['min_total_orders']:,}"
    validation_passed_business = False
print_check_result("TOTAL ORDERS (>= 3M)", status, msg)

# 2. Total customers validation
print("\n2. Business Rule - Total Customers:")
total_customers = summary_row["total_customers"]
if total_customers >= expected_metrics["min_total_customers"]:
    status = "PASS"
    msg = f"Total customers ({total_customers:,}) >= {expected_metrics['min_total_customers']:,}"
else:
    status = "FAIL"
    msg = f"Total customers ({total_customers:,}) < {expected_metrics['min_total_customers']:,}"
    validation_passed_business = False
print_check_result("TOTAL CUSTOMERS (>= 200K)", status, msg)

# 3. Reorder rate validation
print("\n3. Business Rule - Reorder Rate:")
reorder_rate = summary_row["overall_reorder_rate"]
min_rate = expected_metrics["min_reorder_rate"]
max_rate = expected_metrics["max_reorder_rate"]
if min_rate <= reorder_rate <= max_rate:
    status = "PASS"
    msg = f"Reorder rate ({reorder_rate}%) within expected range [{min_rate}%, {max_rate}%]"
else:
    status = "FAIL"
    msg = f"Reorder rate ({reorder_rate}%) outside expected range [{min_rate}%, {max_rate}%]"
    validation_passed_business = False
print_check_result("REORDER RATE (50-70%)", status, msg)

# 4. Avg basket size validation
print("\n4. Business Rule - Avg Basket Size:")
avg_basket = summary_row["avg_basket_size"]
min_basket = expected_metrics["min_avg_basket_size"]
max_basket = expected_metrics["max_avg_basket_size"]
if min_basket <= avg_basket <= max_basket:
    status = "PASS"
    msg = f"Avg basket size ({avg_basket}) within expected range [{min_basket}, {max_basket}]"
else:
    status = "FAIL"
    msg = f"Avg basket size ({avg_basket}) outside expected range [{min_basket}, {max_basket}]"
    validation_passed_business = False
print_check_result("AVG BASKET SIZE (8-12)", status, msg)

print("\n" + "="*60)
if validation_passed_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results
validation_passed = validation_passed_technical and validation_passed_business

print("\n" + "="*60)
print("OVERALL VALIDATION RESULT")
print("="*60)
print(f"  Technical Validation: {'PASSED ✓' if validation_passed_technical else 'FAILED ✗'}")
print(f"  Business Validation:  {'PASSED ✓' if validation_passed_business else 'FAILED ✗'}")
print("="*60)

if validation_passed:
    print(f"\n✓ ALL VALIDATIONS PASSED - Ready to persist to Gold layer")
else:
    print(f"\n✗ SOME VALIDATIONS FAILED - Will NOT persist")
    print("\nPlease review and fix the errors above before re-running.")

print("="*60)

In [0]:
# Only persist if validation passed
if validation_passed:
    print("Persisting to Gold layer...")
    
    executive_summary_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{target_table}")
    
    # Verify
    final_count = spark.table(f"{gold_schema}.{target_table}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Executive Summary table persisted to Gold layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {gold_schema}.{target_table}")
    print(f"  Rows: {final_count:,}")
    print(f"  Type: Aggregated executive KPIs")
    print(f"  Format: Delta")
    print(f"\nNext Step: Run other Gold layer notebooks for detailed analytics")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")